# H18a: Empirical Depth-Stability of Muon's Learning Rate in a Deep-Linear Toy Model

**Counterpart file:** `run_experiment.py`

This notebook is now a presentation and analysis layer built on top of the script's
shared `run_experiment()` API. It does **not** reimplement the experiment core.
Instead, it imports the script, runs the same computation path, and then presents
the returned structured results with tables, figures, and a calibrated conclusion.

## Scope and claim discipline

The study remains the same deep-linear 32x32 learning-rate experiment across depths
`[2, 4, 8, 16]`, but the interpretation here is deliberately narrower than a proof claim.
The notebook measures and discusses only:

- initialization-time max layerwise $\|G\|_{op}$
- initialization-time max layerwise $\|\mathrm{ortho}(G)\|_{op}$
- estimated $\lambda_{\max}(H_{init})$
- empirical best learning rates from fixed-budget sweeps
- empirical max stable learning rates from short-horizon non-divergence searches
- descriptive log-log fits across depth

The results are therefore **evidence inside this deep-linear toy model**, not a universal
proof about all architectures, tasks, or optimizer behavior.


---
## 1. Imports and notebook-safe access to the shared experiment script

The cells below avoid notebook-hostile patterns such as `__file__` and `sys.exit(...)`.
Instead, they locate `run_experiment.py` explicitly from the current working directory or
one of its parents, then load it as a module.


In [ ]:
import os
import json
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

plt.style.use('default')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)
np.set_printoptions(suppress=True)


In [ ]:
EXPERIMENT_RELATIVE = Path('experiments/Tier_1_Core_Mechanism_Tests/H18a_LR_STABILITY_MECHANISM')

def locate_experiment_dir(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        repo_style = candidate / EXPERIMENT_RELATIVE / 'run_experiment.py'
        if repo_style.exists():
            return repo_style.parent
        local_style = candidate / 'run_experiment.py'
        if local_style.exists() and candidate.name == EXPERIMENT_RELATIVE.name:
            return candidate
    raise FileNotFoundError(
        'Could not locate experiments/Tier_1_Core_Mechanism_Tests/'
        'H18a_LR_STABILITY_MECHANISM/run_experiment.py from the current working directory.'
    )

EXPERIMENT_DIR = locate_experiment_dir(Path.cwd())
RUN_EXPERIMENT_PATH = EXPERIMENT_DIR / 'run_experiment.py'

spec = importlib.util.spec_from_file_location('h18a_run_experiment', RUN_EXPERIMENT_PATH)
h18a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h18a)

display(Markdown(
    f"**Experiment directory:** `{EXPERIMENT_DIR}`  \n"
    f"**Imported module:** `{RUN_EXPERIMENT_PATH.name}`  \n"
    f"**Current working directory:** `{Path.cwd().resolve()}`"
))


---
## 2. Reproducibility, runtime mode, and execution

- Default behavior is the **full** experiment, matching the script's default configuration.
- For quick verification, set environment variable `H18A_RUN_MODE=smoke` before execution.
- `H18A_VERBOSE=1` keeps the script's phase-level progress prints; `0` makes the run quieter.

The full run can take **minutes on CPU**, because phase 2 alone performs many training runs
over dense learning-rate grids.


In [ ]:
RUN_MODE = os.environ.get('H18A_RUN_MODE', 'full').strip().lower()
VERBOSE = os.environ.get('H18A_VERBOSE', '1').strip().lower() not in {'0', 'false', 'no'}

if RUN_MODE == 'smoke':
    config = h18a.make_smoke_test_config()
elif RUN_MODE == 'full':
    config = h18a.resolve_config()
else:
    raise ValueError("H18A_RUN_MODE must be 'full' or 'smoke'.")

print(f'Run mode: {RUN_MODE}')
print(f'Verbose script logging: {VERBOSE}')
print('Starting shared run_experiment() path...')
results = h18a.run_experiment(config=config, verbose=VERBOSE)
print(f"Completed in {results['metadata']['runtime_sec']:.2f} seconds.")


In [ ]:
metadata_df = pd.DataFrame([
    {
        'field': 'experiment_id',
        'value': results['metadata']['experiment_id'],
    },
    {
        'field': 'title',
        'value': results['metadata']['title'],
    },
    {
        'field': 'scope_statement',
        'value': results['metadata']['scope_statement'],
    },
    {
        'field': 'timestamp_utc',
        'value': results['metadata']['timestamp_utc'],
    },
    {
        'field': 'runtime_sec',
        'value': round(results['metadata']['runtime_sec'], 3),
    },
])

work_estimate_df = pd.DataFrame([results['work_estimate']]).T.reset_index()
work_estimate_df.columns = ['quantity', 'value']

config_df = pd.DataFrame([
    {
        'field': key,
        'value': json.dumps(value) if isinstance(value, (list, dict)) else value,
    }
    for key, value in results['config'].items()
])

display(Markdown('### Reproducibility and runtime log'))
display(metadata_df)
display(Markdown('### Estimated work for this configuration'))
display(work_estimate_df)
display(Markdown('### Exact configuration used in this notebook run'))
display(config_df)


---
## 3. Convert structured results into analysis tables

The script returns nested dictionaries for raw per-seed results and summaries. The helper
functions below flatten those outputs into `pandas` tables for plotting and inspection.


In [ ]:
def to_phase1_seed_df(results_dict):
    rows = []
    for depth in results_dict['config']['depths']:
        for record in results_dict['phase1']['by_depth'][depth]['per_seed']:
            rows.append({
                'depth': depth,
                'seed': record['seed'],
                'max_grad_op_norm': record['max_grad_op_norm'],
                'max_ortho_grad_op_norm': record['max_ortho_grad_op_norm'],
                'lambda_max_init': record['lambda_max_init'],
            })
    return pd.DataFrame(rows).sort_values(['depth', 'seed']).reset_index(drop=True)

def to_phase2_sweep_df(results_dict):
    rows = []
    for depth in results_dict['config']['depths']:
        for optimizer, block in results_dict['phase2']['by_depth'][depth].items():
            for record in block['per_seed']:
                for lr, final_loss, step1_max, is_finite in zip(
                    record['lr_grid'],
                    record['final_losses'],
                    record['step1_max_op_norms'],
                    record['finite_mask'],
                ):
                    rows.append({
                        'depth': depth,
                        'optimizer': optimizer,
                        'seed': record['seed'],
                        'lr': lr,
                        'final_loss': final_loss,
                        'step1_max_op_norm': step1_max,
                        'is_finite': bool(is_finite),
                    })
    return pd.DataFrame(rows).sort_values(['optimizer', 'depth', 'seed', 'lr']).reset_index(drop=True)

def to_phase2_best_df(results_dict):
    rows = []
    for depth in results_dict['config']['depths']:
        for optimizer, block in results_dict['phase2']['by_depth'][depth].items():
            for record in block['per_seed']:
                rows.append({
                    'depth': depth,
                    'optimizer': optimizer,
                    'seed': record['seed'],
                    'best_lr': record['best_lr'],
                    'best_loss': record['best_loss'],
                    'best_step1_max_op_norm': record['best_step1_max_op_norm'],
                    'found_finite_run': record['found_finite_run'],
                    'best_lr_on_grid_edge': record['best_lr_on_grid_edge'],
                    'num_finite_runs': record['num_finite_runs'],
                })
    return pd.DataFrame(rows).sort_values(['optimizer', 'depth', 'seed']).reset_index(drop=True)

def to_phase3_df(results_dict):
    rows = []
    for depth in results_dict['config']['depths']:
        for optimizer, block in results_dict['phase3']['by_depth'][depth].items():
            for record in block['per_seed']:
                rows.append({
                    'depth': depth,
                    'optimizer': optimizer,
                    'seed': record['seed'],
                    'max_stable_lr': record['max_stable_lr'],
                })
    return pd.DataFrame(rows).sort_values(['optimizer', 'depth', 'seed']).reset_index(drop=True)

def tests_to_df(results_dict):
    rows = []
    for test_name, test in results_dict['tests'].items():
        rows.append({
            'test': test_name,
            'pass': test['pass'],
            'value': test['value'],
            'criterion': f"{test['criterion']} {test['threshold']}",
            'measured_quantity': test['measured_quantity'],
            'description': test['description'],
        })
    return pd.DataFrame(rows).sort_values('test').reset_index(drop=True)

def finite_median(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.median(arr))

def finite_mean(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.mean(arr))

def jitter_positions(center, n, half_width=0.12):
    if n <= 1:
        return np.array([center], dtype=float)
    return center + np.linspace(-half_width, half_width, n)


In [ ]:
phase1_seed_df = to_phase1_seed_df(results)
phase2_sweep_df = to_phase2_sweep_df(results)
phase2_best_df = to_phase2_best_df(results)
phase3_df = to_phase3_df(results)
summary_df = pd.DataFrame(results['summary_table']).sort_values('depth').reset_index(drop=True)
fits_df = pd.DataFrame(results['fits'])[[
    'metric', 'slope', 'r2', 'ratio_low_over_high', 'depth_low', 'depth_high'
]]
tests_df = tests_to_df(results)
grid_df = pd.DataFrame(results['grid_diagnostics']['rows']).sort_values('depth').reset_index(drop=True)

summary_display = summary_df.copy()
for col in summary_display.columns:
    if col != 'depth':
        summary_display[col] = summary_display[col].map(lambda x: round(x, 6) if pd.notna(x) else x)

display(Markdown('### Compact depth-wise summary table'))
display(summary_display)


---
## 4. Phase 1: initialization measurements

These plots show exactly what the experiment measures at initialization:

- max layerwise $\|G\|_{op}$ per seed
- max layerwise $\|\mathrm{ortho}(G)\|_{op}$ per seed
- estimated $\lambda_{\max}(H_{init})$ per seed

This is descriptive evidence about the deep-linear setup at initialization, not a full dynamical proof.


In [ ]:
depths = results['config']['depths']
color_positions = np.linspace(0.15, 0.85, len(depths)) if len(depths) > 1 else [0.5]
depth_colors = {depth: plt.cm.viridis(pos) for depth, pos in zip(depths, color_positions)}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
metric_specs = [
    ('max_grad_op_norm', 'max $||G||_{op}$ at initialization', 'Initialization gradient operator norms', 'log'),
    ('max_ortho_grad_op_norm', 'max $||ortho(G)||_{op}$ at initialization', 'Orthogonalized gradient operator norms', 'linear'),
    ('lambda_max_init', 'estimated $λ_{max}(H_{init})$', 'Initialization Hessian spectral estimates', 'log'),
]

for ax, (column, ylabel, title, yscale) in zip(axes, metric_specs):
    for depth in depths:
        subset = phase1_seed_df[phase1_seed_df['depth'] == depth]
        x = jitter_positions(depth, len(subset))
        ax.scatter(x, subset[column], color=depth_colors[depth], alpha=0.8, s=50)
    grouped = phase1_seed_df.groupby('depth')[column].agg(['mean', 'std']).reset_index()
    ax.errorbar(
        grouped['depth'], grouped['mean'], yerr=grouped['std'],
        color='black', marker='o', linewidth=1.5, capsize=4, label='mean ± std'
    )
    if column == 'max_ortho_grad_op_norm':
        ax.axhline(1.0, color='tab:red', linestyle='--', linewidth=1.2, label='reference = 1')
    ax.set_title(title)
    ax.set_xlabel('depth')
    ax.set_ylabel(ylabel)
    ax.set_xticks(depths)
    if yscale == 'log':
        ax.set_yscale('log')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()


---
## 5. Phase 2: empirical best learning rates from LR sweeps

The next figure uses the script's raw sweep outputs. For each optimizer and depth, it plots
the **median finite final loss** over seeds as a function of learning rate, then marks the
median selected best learning rate for that depth.

This keeps the notebook honest about what is actually optimized: a fixed-budget training loss
after `train_steps` updates, not an analytic optimum.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), sharey=True)

for ax, optimizer in zip(axes, ['sgd', 'muon']):
    optimizer_best = phase2_best_df[phase2_best_df['optimizer'] == optimizer]
    for depth in depths:
        subset = phase2_sweep_df[
            (phase2_sweep_df['optimizer'] == optimizer) &
            (phase2_sweep_df['depth'] == depth)
        ]
        curve_rows = []
        for lr, lr_block in subset.groupby('lr'):
            finite_losses = lr_block.loc[lr_block['is_finite'], 'final_loss'].to_numpy(dtype=float)
            curve_rows.append({
                'lr': float(lr),
                'median_finite_final_loss': finite_median(finite_losses),
                'finite_fraction': float(lr_block['is_finite'].mean()),
            })
        curve_df = pd.DataFrame(curve_rows).sort_values('lr')
        color = depth_colors[depth]
        ax.plot(
            curve_df['lr'], curve_df['median_finite_final_loss'],
            marker='o', linewidth=1.8, color=color, label=f'depth {depth}'
        )

        best_lr = finite_median(optimizer_best.loc[optimizer_best['depth'] == depth, 'best_lr'])
        best_loss = finite_median(optimizer_best.loc[optimizer_best['depth'] == depth, 'best_loss'])
        if np.isfinite(best_lr) and np.isfinite(best_loss):
            ax.axvline(best_lr, color=color, linestyle='--', alpha=0.35)
            ax.scatter([best_lr], [best_loss], color=color, edgecolor='black', s=90, zorder=5)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f"{optimizer.upper()} LR sweep outcomes")
    ax.set_xlabel('learning rate')
    ax.set_ylabel('median finite final loss' if optimizer == 'sgd' else '')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()


---
## 6. Depth dependence of best LR and max stable LR

The left panel summarizes the **best LR selected from phase-2 sweeps**. The right panel
summarizes the **empirical max stable LR** from the short-horizon binary search in phase 3.

Both panels are empirical summaries, not theoretical boundaries.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), sharex=True)
optimizer_styles = {
    'sgd': {'color': 'tab:blue', 'marker': 'o', 'label': 'SGD'},
    'muon': {'color': 'tab:orange', 'marker': 's', 'label': 'Muon'},
}

for optimizer, style in optimizer_styles.items():
    subset = phase2_best_df[phase2_best_df['optimizer'] == optimizer]
    for depth in depths:
        seed_block = subset[subset['depth'] == depth].sort_values('seed')
        x = jitter_positions(depth + (-0.12 if optimizer == 'sgd' else 0.12), len(seed_block), half_width=0.03)
        axes[0].scatter(x, seed_block['best_lr'], color=style['color'], alpha=0.55, s=45)
    grouped = subset.groupby('depth')['best_lr'].median().reset_index()
    axes[0].plot(grouped['depth'], grouped['best_lr'], color=style['color'], marker=style['marker'], linewidth=2, label=style['label'])

for optimizer, style in optimizer_styles.items():
    subset = phase3_df[phase3_df['optimizer'] == optimizer]
    for depth in depths:
        seed_block = subset[subset['depth'] == depth].sort_values('seed')
        x = jitter_positions(depth + (-0.12 if optimizer == 'sgd' else 0.12), len(seed_block), half_width=0.03)
        axes[1].scatter(x, seed_block['max_stable_lr'], color=style['color'], alpha=0.55, s=45)
    grouped = subset.groupby('depth')['max_stable_lr'].median().reset_index()
    axes[1].plot(grouped['depth'], grouped['max_stable_lr'], color=style['color'], marker=style['marker'], linewidth=2, label=style['label'])

axes[0].set_title('Empirical best LR from phase-2 sweeps')
axes[1].set_title('Empirical max stable LR from phase-3 search')

for ax in axes:
    ax.set_xlabel('depth')
    ax.set_ylabel('learning rate')
    ax.set_xticks(depths)
    ax.set_yscale('log')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()


---
## 7. Summary tables, depth-scaling fits, and T1–T5 checks

The tables below expose the returned structured summaries directly rather than hiding them in
prose. The T1–T5 table reports the actual measured values and thresholds used in this run.


In [ ]:
fits_display = fits_df.copy()
for col in ['slope', 'r2', 'ratio_low_over_high']:
    fits_display[col] = fits_display[col].map(lambda x: round(x, 6) if pd.notna(x) else x)

tests_display = tests_df.copy()
tests_display['value'] = tests_display['value'].map(lambda x: round(x, 6) if pd.notna(x) else x)

display(Markdown('### Descriptive log-log fit summary'))
display(fits_display)

display(Markdown('### T1–T5 verdict table with actual measured values'))
display(tests_display)

display(Markdown('### LR-grid edge diagnostics'))
display(grid_df)


---
## 8. Calibrated conclusion and caveats

The conclusion below is generated from the returned results. It is intentionally careful:
it summarizes the evidence in this run without claiming a universal proof.


In [ ]:
low_depth = results['config']['reference_depth_low']
high_depth = results['config']['reference_depth_high']
verdict = results['verdict']
grid_note = (
    'At least one best LR landed on a grid boundary, so follow-up runs should consider widening the sweep grid.'
    if results['grid_diagnostics']['any_best_lr_on_edge']
    else 'No best LR landed on a grid boundary in this run.'
)
failed_text = ', '.join(verdict['failed_tests']) if verdict['failed_tests'] else 'none'

conclusion_md = f"""
### Conclusion

- **Checks passed:** {verdict['n_pass']} / {verdict['n_tests']}
- **Reference depths:** {low_depth} and {high_depth}
- **SGD median best-LR drop:** {verdict['sgd_best_lr_drop_ratio']:.2f}x
- **Muon median best-LR drop:** {verdict['muon_best_lr_drop_ratio']:.2f}x
- **Relative drop advantage (SGD / Muon):** {verdict['relative_drop_advantage']:.2f}x
- **Failed checks:** {failed_text}

**Interpretation.** In this deep-linear toy model, the measured results are consistent with
the idea that operator-norm clamping helps explain why Muon's empirically good learning rate
varies less with depth than SGD's. The strongest direct measurements here are initialization-time
operator norms, fixed-budget LR-sweep optima, short-horizon empirical stability thresholds, and
descriptive depth-scaling fits.

**What this notebook does not show.** It does not provide a full proof, a complete causal chain,
or a universal statement about nonlinear architectures, realistic datasets, or practical training
recipes beyond this toy setting.

**Grid diagnostic.** {grid_note}

**Scope statement from the script.** {results['metadata']['scope_statement']}
"""

display(Markdown(conclusion_md))
